In [ ]:
import pandas as pd  
import numpy as np  
import matplotlib.pyplot as plt 
from collections import Counter
import re
import seaborn as sns 

In [ ]:
df = pd.read_csv("startup_funding.csv")

In [ ]:
df = df.set_index('Sr No')
df = df.drop(columns='Remarks')
df = df.rename(columns={
    "Date dd/mm/yyyy":"Date",
    "Startup Name":"Start Up",
    "Industry Vertical":"Vertical",
    "City  Location":"City",
    "Investors Name":"Investors",
    "InvestmentnType":"Round",
    "Amount in USD":"Amount"
})
def change(x):
    s = x.split("/")
    if len(s[0]) < 3 and int(s[0]) > 12:
        temp = s[1]
        s[1] = s[0]
        s[0] = temp
    return "/".join(s)

def USDtoINR(x):
    return (x*93.37)/10000000

df['Date']  = df['Date'].apply(lambda x: change(x))
df['Date'] = df['Date'].str.replace("05/072018", "05/07/2018")
df['Date'] = df['Date'].str.replace("01/07/015", "05/07/2015")
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df.sample(4)

df['Amount'] = df['Amount'].fillna('0')
df['Amount'] = df['Amount'].str.replace(",", "")
df = df[df['Amount'].str.isdigit()]
df['Amount'] = df['Amount'].astype('float')
df['Amount'] = df['Amount'].apply(lambda x: USDtoINR(x))
df = df.dropna(subset=['Date','Vertical', 'City', 'Investors', 'Round'])
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month

# little data cleaning
# city name correction
city_map = {
    'Bengaluru': 'Bangalore',
    'Gurugram':  'Gurgaon',
    'Delhi':     'New Delhi',
    'New delhi': 'New Delhi',
}
df['City'] = df['City'].str.strip().replace(city_map)
# vertical column correction 
vertical_map = {
    'ECommerce':           'eCommerce',
    'E-Commerce':          'eCommerce',
    'E-commerce':          'eCommerce',
    'Ecommerce':           'eCommerce',
    'FinTech':             'Finance',
    'Fin-Tech':            'Finance',
    'Ed-Tech':             'Education',
    'Health and Wellness': 'Healthcare',
    'Food and Beverage':   'Food & Beverage',
}
df['Vertical'] = df['Vertical'].str.strip().replace(vertical_map)
df['Start Up'] = df['Start Up'].str.strip() # to remove that extra whitepsaces that appears before the names of startups
df['SubVertical'] = df['SubVertical'].fillna('Unknown') 

def normalize_round(r):
    r = r.strip().lower()
    if 'angel' in r and 'seed' not in r:
        return 'Angel Funding'
    elif 'seed' in r and 'angel' not in r:
        return 'Seed Funding'
    elif 'seed' in r and 'angel' in r:
        return 'Seed/Angel Funding'  # only when both are explicitly mentioned together
    elif 'private equity' in r or 'privateequity' in r:
        return 'Private Equity'
    elif 'debt' in r:
        return 'Debt Funding'
    elif 'series a' in r or 'pre-series a' in r or 'pre series a' in r:
        return 'Series A'
    elif 'series b' in r:
        return 'Series B'
    elif 'series c' in r:
        return 'Series C'
    elif 'series d' in r:
        return 'Series D'
    elif 'series e' in r:
        return 'Series E'
    elif 'series f' in r:
        return 'Series F'
    elif 'venture' in r:
        return 'Venture Round'
    else:
        return r.title()

df['Round'] = df['Round'].apply(normalize_round)



def normalize_city(r):
    indian_states = [
        'Bangalore', 'Bengaluru', 'Mumbai', 'Delhi', 'New Delhi', 'Pune',
        'Hyderabad', 'Chennai', 'Gurgaon', 'Gurugram', 'Noida', 'Kolkata',
        'Ahmedabad', 'Jaipur', 'Indore', 'Chandigarh', 'Vadodara', 'Goa',
        'Coimbatore', 'Surat', 'Nagpur', 'Kochi', 'India'
    ]
    us_states = [
        'US', 'USA', 'SFO', 'San Francisco', 'Seattle', 'Houston', 'California',
        'Palo Alto', 'San Mateo', 'New York', 'NY', 'Dallas', 'Boston',
        'Chicago', 'Los Angeles', 'Austin', 'Denver', 'Atlanta'
    ]
    
    r = r.lower()
    
    if 'us' in r:
        return 'India/USA'
    
    for state in indian_states:
        if state.lower() in r:
            for us_state in us_states:
                if us_state.lower() in r:
                    return 'India/USA'
            
    for state in indian_states:
        if state.lower() in r and ('singapore' in r or 'sg' in r):
                    return 'India/Singapore'
                
    return r.title()

#  ~~~~~~~~~~~~~~~~~~~~~ AI doing Normalization on start up colums ~~~~~~~~~~~~~~~~~~~~~~~
df['City'] = df['City'].apply(normalize_city)

# 1. Strip leading/trailing whitespace
df['Start Up'] = df['Start Up'].str.strip()

# 2. Fix encoding garbage (BYJUâS, \xe2\x80\x99 etc. → clean apostrophe)
df['Start Up'] = df['Start Up'].str.replace(r'\\xe2\\x80\\x99', "'", regex=True)
df['Start Up'] = df['Start Up'].str.replace(r'â\x80\x99', "'", regex=True)
df['Start Up'] = df['Start Up'].str.replace(r'\\"', '', regex=True)  # remove escaped quotes

# 3. Remove legal suffixes (Pvt. Ltd, Ltd., Inc, Technologies etc.)
suffixes = r'\s*(Pvt\.?\s*Ltd\.?|Ltd\.?|Inc\.?|Technologies\s*Pvt\.?\s*Ltd\.?|Pvt\.?)$'
df['Start Up'] = df['Start Up'].str.replace(suffixes, '', regex=True, flags=re.IGNORECASE).str.strip()

# 4. Remove full URLs (https://www.wealthbucket.in/)
df['Start Up'] = df['Start Up'].apply(
    lambda x: re.sub(r'https?://\S+', '', x).strip() if isinstance(x, str) else x
)

# 5. Remove .com / .in / .io / .ai domain extensions from names
# e.g. "Lenskart.com" → "Lenskart", "Housing.com" → "Housing"
df['Start Up'] = df['Start Up'].str.replace(r'\.(com|in|io|ai|co)$', '', regex=True, flags=re.IGNORECASE)

# 6. Standardize to Title Case
df['Start Up'] = df['Start Up'].str.title()

# 7. Manual fixes for known bad entries
manual_map = {
    "Byju'S":  "BYJU'S",
    "Byju\\Xe2\\X80\\X99S": "BYJU'S",
}
df['Start Up'] = df['Start Up'].replace(manual_map)


In [58]:
df = df[~(df['Start Up'] == '')]
investors = df['Investors'].str.strip().str.split(", ").explode()
# df.to_csv('startup_cleaned.csv')
investors_count = investors.value_counts()
mask = (investors_count > 2)
investors_count = investors_count[mask]
investors_count.tail(60)

Investors
Info Edge                                                    3
Sol Primero                                                  3
Kashyap Deorah                                               3
GVFL                                                         3
Vy Capital                                                   3
Harsh Mariwala                                               3
Agnus Capital                                                3
Arun Tadanki                                                 3
Ankit Nagori                                                 3
SIDBI Venture Capital Ltd                                    3
Axis Capital                                                 3
Ankur Capital                                                3
Ankit Bhati                                                  3
Multiple investors through Ten Minute Million competition    3
Raman Roy                                                    3
Greenoaks Capital                            

In [ ]:
# similar_investors = list(df[df['Investors'].str.contains('IDG Ventures India')]['Investors'].str.strip(" ").str.split(", ").sum())
# similar_investors

In [ ]:
# c = Counter(similar_investors)
# c.pop('IDG Ventures India')
# c = sorted(c.items(), key=lambda x: x[1], reverse=True)
# type(c[2][0])


In [ ]:
# df.groupby('Start Up')['Vertical'].count().sort_values(ascending=False)
# df[df['Start Up'] == "Swiggy"]
# df.groupby('Start Up')['Amount'].sum().max()

In [ ]:
# mom_total_funding = df.groupby(['year', 'month'])['Amount'].sum().reset_index()
# mom_no_of_startups = df.groupby(['year', 'month'])['Start Up'].count().reset_index()
# mom_total_funding['x_axis'] = mom_total_funding['year'].astype(str) + '/' + mom_total_funding['month'].astype(str)
# mom_no_of_startups['x_axis'] = mom_no_of_startups['year'].astype(str) + '/' + mom_no_of_startups['month'].astype(str)

# mom_no_of_startups.drop(columns=['year', 'month'], inplace=True)
# mom_no_of_startups.rename(columns={'Start Up': 'y_axis'}, inplace=True)

# mom_total_funding.drop(columns=['year', 'month'], inplace=True)
# mom_total_funding.rename(columns={'Amount': 'y_axis'}, inplace=True)

# mom_total_funding

In [ ]:
# type_of_funding = df.groupby("Round")['Amount'].sum().head(490)
# city_funding = df.groupby("City")['Amount'].sum().head(490)
# city_funding = np.round(city_funding.sort_values(ascending=False), 2)
# type_of_funding = np.round(type_of_funding.sort_values(ascending=False), 2)
# city_funding = city_funding[city_funding > 0]
# type_of_funding = type_of_funding[type_of_funding > 0]
# type_of_funding

In [ ]:

# startups = df.groupby(['Start Up', 'year'])['Amount'].sum().sort_values(ascending=False)
# startups = startups[startups > 0]
# startups.sort_index(level=[1, 0], ascending=[True, True])

In [ ]:
# investor_df = df['Investors'].str.strip().str.split(", ").explode().dropna()
# investor_df = investor_df[investor_df.str.match(r'^$|unknown|undisclosed|')]
# investor_df = investor_df.value_counts().head(5).reset_index().rename(columns={
#     "count":"Total Investments"
# })

In [ ]:
# top_sector_df = df.groupby(['Vertical'])['Amount'].sum().sort_values(ascending=False).head(10)
# top_sector_df = df[df['Vertical'].isin(top_sector_df.index)]
# sector_year_funding = top_sector_df.pivot_table(index='Vertical', columns='year', values='Amount', aggfunc='sum').fillna(0)

# fig, ax = plt.subplots(figsize=(12, 6))
# im = ax.imshow(sector_year_funding, cmap='YlOrRd', aspect='auto')

# for i in range(len(sector_year_funding.index)):
#     for j in range(len(sector_year_funding.columns)):
#         ax.text(j, i, f'{sector_year_funding.iloc[i, j]:0.0f}', 
#                 ha='center', va='center', fontsize=8)
        
# ax.set_xticks(range(len(sector_year_funding.columns)))
# ax.set_xticklabels(sector_year_funding.columns)  # years
# ax.set_yticks(range(len(sector_year_funding.index)))
# ax.set_yticklabels(sector_year_funding.index)        
# fig.colorbar(im, ax=ax, label='Amount (INR cr)')

# ax.set_title("Sector wise Funding Heatmap (2015-2020)")
# ax.set_xlabel("Year")
# ax.set_ylabel("Sector")


In [ ]:
# df[df['Start Up'] == 'Swiggy']

In [ ]:
# startup_vertical = df[df['Start Up'] == 'Swiggy']['Vertical'].mode()[0]
# startup_subvertical = df[df['Start Up'] == 'Swiggy']['SubVertical'].mode()[0]
# startup_city     = df[df['Start Up'] == 'Swiggy']['City'].mode()[0]
# startup_round    = df[df['Start Up'] == 'Swiggy']['Round'].mode()[0]
# df['Start Up'] = df['Start Up'].str.strip()
# similar_startups = df[~(df['Start Up'] == 'Swiggy')]

# similar_startups = df.groupby('Start Up').agg({
#     'Vertical':lambda x: x.mode()[0],
#     'SubVertical': lambda x: x.mode()[0],
#     'Round': lambda x: x.mode()[0],
#     'City': lambda x: x.mode()[0] 
# })
# similar_startups['score'] = (
#     (similar_startups['Vertical'] == startup_vertical).astype(int) +
#     (similar_startups['SubVertical'] == startup_subvertical).astype(int) +
#     (similar_startups['City']     == startup_city).astype(int)     +
#     (similar_startups['Round']    == startup_round).astype(int)
# )
# similar_startups = similar_startups.sort_values(by=['score'], ascending=False).head(3).reset_index()
# similar_startups

In [ ]:
# df[df['Start Up'] == 'nan']

,Date,Start Up,Vertical,SubVertical,City,Investors,Round,Amount,year,month
Sr No,,,,,,,,,,
